# Part 1 — Population Synthesis & Activity Generation

Synthesizes a small population, generates personas, assigns
work/school zones, creates activity lists, and schedules them.

Saves all intermediate state to `../data/processed/pipeline_state.json`
so that **Part 2** can continue from here.

**Requires** an `ANTHROPIC_API_KEY` environment variable.

## 1. Imports + client setup

In [ ]:
from aibm import ZoneSpec, synthesize_population
from aibm.llm import AnthropicClient
from aibm.zone import Zone

client = AnthropicClient()
print("Client ready.")

## 2. Define zones

Three zones for our toy model:
- **north** — residential suburb
- **centre** — commercial downtown
- **east** — mixed: residential, commercial, and school

In [ ]:
zones = [
    Zone(
        id="north",
        name="North Suburb",
        x=0.0,
        y=1.0,
        land_use={
            "residential": True,
            "commercial": False,
            "school": False,
        },
    ),
    Zone(
        id="centre",
        name="City Centre",
        x=0.0,
        y=0.0,
        land_use={
            "residential": False,
            "commercial": True,
            "school": False,
        },
    ),
    Zone(
        id="east",
        name="East Side",
        x=1.0,
        y=0.0,
        land_use={
            "residential": True,
            "commercial": True,
            "school": True,
        },
    ),
]

zone_lookup = {z.id: z for z in zones}
print(f"{len(zones)} zones defined.")

## 3. Define travel times

Dict mapping `(origin, destination)` to `{mode: minutes}`.

In [ ]:
travel_times = {
    ("north", "north"): {
        "car": 5, "transit": 10, "bike": 8, "walk": 15,
    },
    ("north", "centre"): {
        "car": 15, "transit": 25, "bike": 30, "walk": 60,
    },
    ("north", "east"): {
        "car": 20, "transit": 35, "bike": 40, "walk": 80,
    },
    ("centre", "north"): {
        "car": 15, "transit": 25, "bike": 30, "walk": 60,
    },
    ("centre", "centre"): {
        "car": 5, "transit": 8, "bike": 7, "walk": 12,
    },
    ("centre", "east"): {
        "car": 10, "transit": 20, "bike": 25, "walk": 50,
    },
    ("east", "north"): {
        "car": 20, "transit": 35, "bike": 40, "walk": 80,
    },
    ("east", "centre"): {
        "car": 10, "transit": 20, "bike": 25, "walk": 50,
    },
    ("east", "east"): {
        "car": 5, "transit": 10, "bike": 8, "walk": 15,
    },
}


def get_travel_times_from(
    home: str,
) -> dict[str, dict[str, float]]:
    """Get travel times from a home zone to all zones."""
    return {
        dest: modes
        for (orig, dest), modes in travel_times.items()
        if orig == home
    }


print("Travel times defined for all zone pairs.")

## 4. Synthesize ~10 agents

Two small zones, a few households each.

In [ ]:
zone_specs = [
    ZoneSpec(
        zone_id="north",
        n_households=2,
        household_size_dist={1: 0.2, 2: 0.5, 3: 0.3},
        age_dist={"0-17": 0.15, "18-64": 0.70, "65+": 0.15},
        employment_rate=0.70,
        student_rate=0.10,
        vehicle_dist={0: 0.1, 1: 0.6, 2: 0.3},
        license_rate=0.85,
    ),
    ZoneSpec(
        zone_id="east",
        n_households=2,
        household_size_dist={1: 0.3, 2: 0.4, 3: 0.3},
        age_dist={"0-17": 0.10, "18-64": 0.75, "65+": 0.15},
        employment_rate=0.50,
        student_rate=0.25,
        vehicle_dist={0: 0.4, 1: 0.45, 2: 0.15},
        license_rate=0.65,
    ),
]

households = synthesize_population(zone_specs, seed=42)
all_agents = [
    agent for hh in households for agent in hh.members
]

for agent in all_agents:
    agent.model = "claude-haiku-4-5-20251001"

print(
    f"Households: {len(households)}, "
    f"Agents: {len(all_agents)}"
)
for hh in households:
    print(
        f"  {hh}  zone={hh.home_zone}  "
        f"vehicles={hh.num_vehicles}"
    )
    for a in hh.members:
        lic = "licence" if a.has_license else "no licence"
        print(
            f"    {a.name}  age={a.age}  "
            f"{a.employment}  {lic}"
        )

## 5. Generate personas

In [ ]:
hh_lookup = {
    a.id: hh
    for hh in households
    for a in hh.members
}

for agent in all_agents:
    hh = hh_lookup[agent.id]
    agent.generate_persona(client=client, household=hh)
    print(f"{agent.name}: {agent.persona}")

## 6. Choose work/school zones

In [ ]:
for agent in all_agents:
    home = agent.home_zone
    assert home is not None
    tt = get_travel_times_from(home)

    if agent.employment == "employed":
        zone_id = agent.choose_work_zone(
            zones, tt, client=client
        )
        print(f"{agent.name} works in: {zone_id}")
    elif agent.employment == "student":
        zone_id = agent.choose_school_zone(
            zones, tt, client=client
        )
        print(f"{agent.name} studies in: {zone_id}")
    else:
        print(
            f"{agent.name}: no work/school zone needed "
            f"({agent.employment})"
        )

## 7. Generate activities

In [ ]:
agent_activities: dict[str, list] = {}

for agent in all_agents:
    activities = agent.generate_activities(client=client)
    agent_activities[agent.id] = activities
    types = [a.type for a in activities]
    print(f"{agent.name}: {types}")

## 8. Schedule activities

In [ ]:
from aibm.day_plan import DayPlan

agent_plans: dict[str, DayPlan] = {}

for agent in all_agents:
    activities = agent_activities[agent.id]
    day_plan = agent.schedule_activities(
        activities, client=client
    )
    agent_plans[agent.id] = day_plan
    print(f"\n{agent.name}'s schedule:")
    for act in day_plan.activities:
        start = (
            f"{act.start_time:6.0f}"
            if act.start_time is not None
            else "   N/A"
        )
        end = (
            f"{act.end_time:6.0f}"
            if act.end_time is not None
            else "   N/A"
        )
        print(f"  {act.type:20s}  {start}-{end}")

## 9. Save intermediate results

Write all state to JSON so that **Part 2** can pick up
from here without re-running the LLM calls above.

In [ ]:
import json
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

state = {
    "zones": [
        {
            "id": z.id,
            "name": z.name,
            "x": z.x,
            "y": z.y,
            "land_use": z.land_use,
        }
        for z in zones
    ],
    "travel_times": {
        f"{orig},{dest}": modes
        for (orig, dest), modes in travel_times.items()
    },
    "households": [
        {
            "id": hh.id,
            "home_zone": hh.home_zone,
            "num_vehicles": hh.num_vehicles,
            "income_level": hh.income_level,
            "members": [
                {
                    "id": a.id,
                    "name": a.name,
                    "model": a.model,
                    "age": a.age,
                    "employment": a.employment,
                    "has_license": a.has_license,
                    "home_zone": a.home_zone,
                    "work_zone": a.work_zone,
                    "school_zone": a.school_zone,
                    "persona": a.persona,
                }
                for a in hh.members
            ],
        }
        for hh in households
    ],
    "agent_activities": {
        agent_id: [
            {
                "type": act.type,
                "location": act.location,
                "start_time": act.start_time,
                "end_time": act.end_time,
                "is_flexible": act.is_flexible,
            }
            for act in activities
        ]
        for agent_id, activities
        in agent_activities.items()
    },
    "agent_plans": {
        agent_id: [
            {
                "type": act.type,
                "location": act.location,
                "start_time": act.start_time,
                "end_time": act.end_time,
                "is_flexible": act.is_flexible,
            }
            for act in plan.activities
        ]
        for agent_id, plan in agent_plans.items()
    },
}

output_path = output_dir / "pipeline_state.json"
output_path.write_text(json.dumps(state, indent=2))
print(f"Saved pipeline state to {output_path}")